# NeSyKGLLM / MetaMine: Fine-tuning with Symbolic Rule Integration
## Enhanced Knowledge Graph Link Prediction with Comprehensive Metrics

**Description:** This notebook fine-tunes LLMs with and without symbolic rules to measure the impact of neuro-symbolic integration.

**Features:**
- Comprehensive evaluation metrics: Precision, Recall, F1 Score, Time
- Configurable parameters via config dictionary
- Results saved to output directory in multiple formats

## Setup: Install Required Packages

In [1]:
!pip install -q bitsandbytes transformers peft accelerate datasets sentencepiece scikit-learn torch trl matplotlib pyyaml

ERROR: Operation cancelled by user

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
^C
Traceback (most recent call last):
  File "/home/master/.pyenv/versions/NeSy/bin/pip", line 8, in <module>
    sys.exit(main())
  File "/home/master/.pyenv/versions/3.10.12/envs/NeSy/lib/python3.10/site-packages/pip/_internal/cli/main.py", line 70, in main
    return command.main(cmd_args)
  File "/home/master/.pyenv/versions/3.10.12/envs/NeSy/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 100, in main
    with self.main_context():
  File "/home/master/.pyenv/versions/3.10.12/lib/python3.10/contextlib.py", line 142, in __exit__
    next(self.gen)
  File "/home/master/.pyenv/versions/3.10.12/envs/NeSy/lib/python3.10/site-packages/pip/_internal/cli/command_context.py", line 19, in main_context
    with self._main_context:
  File "/home/master/.pyenv/versions/3.10.12/lib/python3.10/contextlib.py", line 576, in __exit_

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Login to HuggingFace

In [ ]:
!huggingface-cli login

### Import Libraries

In [ ]:
import pandas as pd
import json
import csv
import random
import re
import os
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import deque, defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from datasets import Dataset
from functools import partial
import bitsandbytes as bnb

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    AutoPeftModelForCausalLM
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("Libraries imported successfully!")

Libraries imported successfully!


## Configuration

**Modify these settings before running the notebook.**

In [ ]:
# ============================================================================
# CONFIGURATION - MODIFY THESE SETTINGS
# ============================================================================

CONFIG = {
    # Data paths
    'data_dir': '/content/drive/MyDrive/KG-LLM/KG/LDM',
    'kg_file': 'train2id.txt',
    'relation_file': 'id2relation.txt',
    'rules_dir': 'rules',
    'output_dir': '/content/drive/MyDrive/KG-LLM/KG/LDM/newoutput',

    # Model configuration
    'model_name': 'meta-llama/Llama-3.2-1B-Instruct',
    'model_alias': 'LLaMA-3.2-1B',

    # Training parameters
    'train_samples': 2000,
    'test_samples': 1000,
    'eval_samples': 2000,
    'max_path_length': 10,
    'num_training_steps': 500,
    'batch_size': 1,
    'gradient_accumulation_steps': 4,
    'learning_rate': 2e-4,
    'warmup_steps': 10,
    'max_seq_length': 512,
    'logging_steps': 50,
    'save_steps': 100,

    # LoRA configuration
    'lora_r': 16,
    'lora_alpha': 64,
    'lora_dropout': 0.1,

    # Rules configuration
    'pca_threshold': 0.3,
    'max_rules_in_context': 3,

    # GPU configuration
    'max_memory_mb': 40960,
    'use_fp16': True,
    'use_bf16': False,

    # Experiment settings
    'skip_base_eval': False,
    'skip_baseline': False,

    # Results configuration
    'save_predictions': True,
    'generate_plots': True,
    'results_file': 'results.json',
    'summary_csv': 'results_summary.csv'
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("Configuration loaded!")
print(f"Data directory: {CONFIG['data_dir']}")
print(f"Output directory: {CONFIG['output_dir']}")
print(f"Model: {CONFIG['model_name']}")

Configuration loaded!
Data directory: /content/drive/MyDrive/KG-LLM/KG/LDM
Output directory: /content/drive/MyDrive/KG-LLM/KG/LDM/newoutput
Model: meta-llama/Llama-3.2-1B-Instruct


## Evaluation Metrics Data Structure

In [ ]:
@dataclass
class EvaluationMetrics:
    """Container for comprehensive evaluation metrics."""
    accuracy: float
    precision: float
    recall: float
    f1_score: float
    eval_size: int
    eval_time_seconds: float
    training_time_seconds: float = 0.0
    y_true: List[bool] = field(default_factory=list)
    y_pred: List[bool] = field(default_factory=list)

    def to_dict(self, include_predictions: bool = False) -> Dict:
        """Convert metrics to dictionary."""
        result = {
            'accuracy': round(self.accuracy, 4),
            'precision': round(self.precision, 4),
            'recall': round(self.recall, 4),
            'f1_score': round(self.f1_score, 4),
            'eval_size': self.eval_size,
            'eval_time_seconds': round(self.eval_time_seconds, 2),
            'training_time_seconds': round(self.training_time_seconds, 2),
        }

        if include_predictions:
            result['y_true'] = self.y_true
            result['y_pred'] = self.y_pred

        return result

    def __str__(self) -> str:
        return (
            f"Precision: {self.precision:.4f} | "
            f"Recall: {self.recall:.4f} | "
            f"F1: {self.f1_score:.4f} | "
            f"Size: {self.eval_size} | "
            f"Time: {self.eval_time_seconds:.2f}s"
        )

print("EvaluationMetrics class defined!")

EvaluationMetrics class defined!


## Preprocess Knowledge Graph Files

In [ ]:
def preprocess_kg_file(input_file, output_file):
    """Preprocess KG file from tab-separated to space-separated format."""
    with open(input_file, 'r') as f:
        lines = f.readlines()
    with open(output_file, 'w') as f:
        f.write(f"{len(lines)}\n")
        for line in lines:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                f.write(f"{parts[0]} {parts[1]} {parts[2]}\n")
    print(f"Preprocessed {len(lines)} triples")
    return len(lines)

# Preprocess the knowledge graph
kg_input = os.path.join(CONFIG['data_dir'], CONFIG['kg_file'])
kg_processed = os.path.join(CONFIG['data_dir'], 'train2id_processed.txt')
preprocess_kg_file(kg_input, kg_processed)
print("Knowledge graph preprocessed!")

Preprocessed 1881747 triples
Knowledge graph preprocessed!


In [ ]:
# Create relation mapping file
relation_file = os.path.join(CONFIG['data_dir'], CONFIG['relation_file'])

relations = set()
with open(kg_processed, 'r') as f:
    n = int(f.readline())
    for line in f:
        parts = line.strip().split()
        if len(parts) == 3:
            relations.add(parts[2])

with open(relation_file, 'w') as f:
    f.write(f"{len(relations)}\n")
    for i, rel in enumerate(sorted(relations)):
        f.write(f"relation_{rel}\t{rel}\n")

print(f"Processed {n} triples with {len(relations)} unique relations")

Processed 1881747 triples with 9 unique relations


## Load Knowledge Graph Functions

In [ ]:
def load_knowledge_graph(file_path):
    """Load knowledge graph from processed file."""
    graph = {}
    nodes = set()
    with open(file_path, 'r') as f:
        num_lines = int(f.readline())
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                node1, node2, relation = parts
                nodes.add(node1)
                nodes.add(node2)
                if node1 not in graph:
                    graph[node1] = {}
                graph[node1][node2] = int(relation)
    return graph, list(nodes)

def load_relation_mapping(file_path):
    """Load relation ID to name mapping."""
    id2relation = {}
    with open(file_path, 'r') as f:
        num_relations = int(f.readline())
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                relation, relation_id = parts
                id2relation[int(relation_id)] = relation
    return id2relation

print("Knowledge graph functions defined!")

Knowledge graph functions defined!


## Symbolic Rule Integration Functions

In [ ]:
def parse_rule_file(file_path):
    """Parse a single rule file and extract rule information."""
    with open(file_path, 'r') as f:
        content = f.read()

    rule_info = {
        'rule_id': None,
        'rule_text': None,
        'head': None,
        'body': None,
        'instances': [],
        'pca_confidence': None,
        'classification': None
    }

    # Extract rule ID and text
    rule_match = re.search(r'Rule (\d+):\s*(.+?)(?=\n\nFormal Rule:)', content, re.DOTALL)
    if rule_match:
        rule_info['rule_id'] = rule_match.group(1)
        rule_info['rule_text'] = rule_match.group(2).strip()

    # Extract head and body predicates
    head_match = re.search(r'Head:\s*(.+)', content)
    body_match = re.search(r'Body:\s*(.+)', content)
    if head_match:
        rule_info['head'] = head_match.group(1).strip()
    if body_match:
        rule_info['body'] = body_match.group(1).strip()

    # Extract instances
    instances_section = re.search(r'Real Instances from Knowledge Graph.*?:\n\n(.+?)(?=\n\nRule Statistics:)', content, re.DOTALL)
    if instances_section:
        for line in instances_section.group(1).strip().split('\n'):
            if line.strip():
                rule_info['instances'].append(line.strip())

    # Extract PCA confidence and classification
    pca_match = re.search(r'PCA Confidence:\s*([\d.]+)', content)
    classification_match = re.search(r'Rule Classification:\s*(\w+)', content)
    if pca_match:
        rule_info['pca_confidence'] = float(pca_match.group(1))
    if classification_match:
        rule_info['classification'] = classification_match.group(1)

    return rule_info

def load_all_rules(rules_directory):
    """Load all rule files from a directory."""
    rules = []
    rule_files = sorted(Path(rules_directory).glob('rule_*.txt'))
    for file_path in rule_files:
        try:
            rule_info = parse_rule_file(file_path)
            rules.append(rule_info)
            print(f"Loaded {file_path.name}: {rule_info['rule_text'][:60] if rule_info['rule_text'] else 'N/A'}...")
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    return rules

def create_rule_context(rules, max_rules=3):
    """Create a textual context from symbolic rules for prompt injection."""
    if not rules:
        return ""
    rule_context = "\n###Symbolic Rules (for reference):\n"
    for i, rule in enumerate(rules[:max_rules], 1):
        if rule.get('rule_text'):
            rule_context += f"{i}. {rule['rule_text']}\n"
            if rule.get('pca_confidence') is not None:
                rule_context += f"   (Confidence: {rule['pca_confidence']:.3f})\n"
    rule_context += "\n"
    return rule_context

print("Symbolic rule functions defined!")

Symbolic rule functions defined!


## Training Data Generation

In [ ]:
def generate_training_data_with_rules(graph, node_list, id2relation, rules,
                                       total_samples=1000, max_path_length=10,
                                       include_reasoning=True, use_rules=True):
    """
    Generate training data with chain-of-thought reasoning.

    Args:
        graph: Knowledge graph dictionary
        node_list: List of nodes
        id2relation: Relation ID to name mapping
        rules: List of symbolic rules
        total_samples: Number of samples to generate
        max_path_length: Maximum path length
        include_reasoning: Whether to include reasoning traces
        use_rules: Whether to include symbolic rule context

    Returns:
        DataFrame with training examples
    """
    data = []
    unique_paths = set()
    pos_count = 0
    neg_count = 0

    rule_context = ""
    if use_rules and rules:
        rule_context = create_rule_context(rules, max_rules=CONFIG['max_rules_in_context'])

    max_attempts = total_samples * 10
    attempts = 0

    while len(data) < total_samples and attempts < max_attempts:
        attempts += 1

        path_length = random.randint(2, max_path_length)
        first_node = random.choice(node_list)
        visited = {first_node}
        path_text = ""
        reasoning_text = ""
        previous_node = first_node

        for step in range(path_length - 1):
            if previous_node not in graph or not graph[previous_node]:
                node = random.choice(node_list)
                safety = 0
                while node in visited and safety < 100:
                    node = random.choice(node_list)
                    safety += 1
                path_text += f'node_{previous_node} not connected with node_{node}. '
                if include_reasoning:
                    reasoning_text += f'node_{previous_node} not connected with node_{node} means there is no relationship. '
                visited.add(node)
                previous_node = node
            else:
                next_node = random.choice(list(graph[previous_node].keys()))
                safety = 0
                while next_node in visited and safety < 100:
                    next_node = random.choice(list(graph[previous_node].keys()))
                    safety += 1

                relation = graph[previous_node][next_node]
                rel_name = id2relation.get(relation, f"relation_{relation}")

                path_text += f'node_{previous_node} has {rel_name} with node_{next_node}. '
                if include_reasoning:
                    reasoning_text += f'node_{previous_node} has {rel_name} with node_{next_node}. '
                visited.add(next_node)
                previous_node = next_node

        last_node = previous_node

        if path_text in unique_paths:
            continue
        unique_paths.add(path_text)

        question = f'Is node_{first_node} connected with node_{last_node}?'
        is_connected = (first_node in graph and last_node in graph[first_node]) or \
                       (last_node in graph and first_node in graph[last_node])

        if is_connected and pos_count >= total_samples // 2:
            continue
        if not is_connected and neg_count >= total_samples // 2:
            continue

        if is_connected:
            if include_reasoning:
                answer = reasoning_text
                if use_rules and rule_context:
                    answer += "Applying symbolic rules and path analysis together: "
                answer += 'The answer is yes.'
            else:
                answer = 'The answer is yes.'
            pos_count += 1
        else:
            if include_reasoning:
                answer = reasoning_text
                if use_rules and rule_context:
                    answer += "Based on symbolic rules and path analysis: "
                answer += 'The answer is no.'
            else:
                answer = 'The answer is no.'
            neg_count += 1

        # Construct prompt
        if include_reasoning:
            if use_rules and rule_context:
                prompt = f"###Instruction:\nAnswer the following yes/no question by reasoning step-by-step. Use the symbolic rules as additional context along with the path information.\n{rule_context}###Input:\n{path_text}{question}\n\n###Response:\n{answer}"
            else:
                prompt = f"###Instruction:\nAnswer the following yes/no question by reasoning step-by-step.\n\n###Input:\n{path_text}{question}\n\n###Response:\n{answer}"
        else:
            if use_rules and rule_context:
                prompt = f"{rule_context}###Input:\n{path_text}{question}\n\n###Response:\n{answer}"
            else:
                prompt = f"###Input:\n{path_text}{question}\n\n###Response:\n{answer}"

        data.append({
            'Prompt': prompt,
            'input_text': path_text + question,
            'output_text': answer,
            'has_rule_context': use_rules and bool(rule_context),
            'is_connected': is_connected
        })

    print(f"Generated {len(data)} samples (positive: {pos_count}, negative: {neg_count})")
    return pd.DataFrame(data)

print("Data generation function defined!")

Data generation function defined!


## Load Graph and Generate Data

In [ ]:
print("="*70)
print("LOADING KNOWLEDGE GRAPH AND SYMBOLIC RULES")
print("="*70)

# Load knowledge graph
graph, node_list = load_knowledge_graph(kg_processed)
id2relation = load_relation_mapping(relation_file)
print(f"Graph loaded: {len(graph)} source nodes, {len(node_list)} total nodes")
print(f"Relations loaded: {len(id2relation)}")

# Load symbolic rules
rules_path = os.path.join(CONFIG['data_dir'], CONFIG['rules_dir'])
rules = []
if os.path.exists(rules_path):
    rules = load_all_rules(rules_path)
    print(f"\nTotal rules loaded: {len(rules)}")
else:
    print(f"\nWarning: Rules directory not found: {rules_path}")
    print("Training will proceed without symbolic rules")

In [ ]:
print("\n" + "="*70)
print("GENERATING TRAINING DATA")
print("="*70)

# Generate training data with rules
print("\n--- Generating training data WITH rules ---")
train_df_with_rules = generate_training_data_with_rules(
    graph, node_list, id2relation, rules,
    total_samples=CONFIG['train_samples'],
    max_path_length=CONFIG['max_path_length'],
    include_reasoning=True,
    use_rules=True
)

# Generate training data without rules
print("\n--- Generating training data WITHOUT rules ---")
train_df_without_rules = generate_training_data_with_rules(
    graph, node_list, id2relation, rules,
    total_samples=CONFIG['train_samples'],
    max_path_length=CONFIG['max_path_length'],
    include_reasoning=True,
    use_rules=False
)

# Generate test data
print("\n--- Generating test data ---")
test_df = generate_training_data_with_rules(
    graph, node_list, id2relation, rules,
    total_samples=CONFIG['test_samples'],
    max_path_length=CONFIG['max_path_length'],
    include_reasoning=True,
    use_rules=True
)

# Save generated data
train_df_with_rules.to_csv(os.path.join(CONFIG['output_dir'], 'train_data_with_rules.csv'), index=False)
train_df_without_rules.to_csv(os.path.join(CONFIG['output_dir'], 'train_data_without_rules.csv'), index=False)
test_df.to_csv(os.path.join(CONFIG['output_dir'], 'test_data.csv'), index=False)

print(f"\n" + "="*70)
print(f"Training samples WITH rules: {len(train_df_with_rules)}")
print(f"Training samples WITHOUT rules: {len(train_df_without_rules)}")
print(f"Test samples: {len(test_df)}")
print("="*70)

## Model Training Functions

In [ ]:
def create_bnb_config():
    """Create BitsAndBytes configuration for 4-bit quantization."""
    compute_dtype = torch.bfloat16 if CONFIG['use_bf16'] else torch.float16
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype
    )

def load_model(model_name, bnb_config):
    """Load model with quantization and tokenizer."""
    n_gpus = torch.cuda.device_count()
    max_memory = f'{CONFIG["max_memory_mb"]}MB'

    print(f"Loading model {model_name} with {n_gpus} GPU(s)")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory={i: max_memory for i in range(n_gpus)} if n_gpus > 0 else None
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

def create_peft_config(modules):
    """Create LoRA configuration."""
    return LoraConfig(
        r=CONFIG['lora_r'],
        lora_alpha=CONFIG['lora_alpha'],
        target_modules=modules,
        lora_dropout=CONFIG['lora_dropout'],
        bias="none",
        task_type="CAUSAL_LM"
    )

def find_all_linear_names(model):
    """Find all linear layer names for LoRA targeting."""
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

def preprocess_batch(batch, tokenizer, max_length=512): 
    """Tokenize a batch of examples."""
    return tokenizer(
        batch["Prompt"],
        truncation=True,
        max_length=max_length,
        padding='max_length'
    )

print("Training functions defined!")

Training functions defined!


## Enhanced Evaluation Function with Precision, Recall, F1, and Time

In [ ]:
def evaluate_model(model, tokenizer, test_df, max_samples=200, device='cuda'):
    """
    Evaluate model with comprehensive metrics.

    Returns:
        EvaluationMetrics object with:
        - Precision, Recall, F1, Accuracy
        - Evaluation time in seconds
    """
    model.eval()
    y_true = []
    y_pred = []

    num_samples = min(len(test_df), max_samples)
    print(f"Evaluating on {num_samples} samples...")

    # Start timing
    start_time = time.time()

    for idx, row in test_df.head(num_samples).iterrows():
        input_text = "###Input:\n" + row['input_text']
        expected_has_yes = 'yes' in row['output_text'].lower()

        inputs = tokenizer(input_text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=150,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False
            )

        model_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
        model_has_yes = 'yes' in model_answer.lower()

        y_true.append(expected_has_yes)
        y_pred.append(model_has_yes)

        if (idx + 1) % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Processed {idx + 1}/{num_samples} samples... ({elapsed:.1f}s elapsed)")

    # End timing
    eval_time = time.time() - start_time

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=True, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=True, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=True, zero_division=0)

    metrics = EvaluationMetrics(
        accuracy=accuracy,
        precision=precision,
        recall=recall,
        f1_score=f1,
        eval_size=num_samples,
        eval_time_seconds=eval_time,
        y_true=y_true,
        y_pred=y_pred
    )

    return metrics

print("Enhanced evaluation function defined!")

Enhanced evaluation function defined!


In [ ]:
def fine_tune_model(model, tokenizer, train_dataset, output_dir, num_steps=500):
    """
    Fine-tune model using LoRA.

    Returns:
        Tuple of (fine-tuned model, training time in seconds)
    """
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    modules = find_all_linear_names(model)
    print(f"Target modules for LoRA: {modules}")

    peft_config = create_peft_config(modules)
    model = get_peft_model(model, peft_config)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")

    training_args = TrainingArguments(
        per_device_train_batch_size=CONFIG['batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
        warmup_steps=CONFIG['warmup_steps'],
        max_steps=num_steps,
        learning_rate=CONFIG['learning_rate'],
        fp16=CONFIG['use_fp16'],
        bf16=CONFIG['use_bf16'],
        logging_steps=CONFIG['logging_steps'],
        output_dir=output_dir,
        optim="paged_adamw_8bit",
        save_strategy="steps",
        save_steps=CONFIG['save_steps'],
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        args=training_args,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
    )

    model.config.use_cache = False

    print("Starting training...")
    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time

    print(f"Training completed in {training_time:.2f} seconds")
    print(f"Saving model to {output_dir}...")
    os.makedirs(output_dir, exist_ok=True)
    trainer.model.save_pretrained(output_dir)

    return model, training_time

print("Fine-tuning function defined!")

Fine-tuning function defined!


## Results Saving and Visualization Functions

In [ ]:
def print_results_table(results, training_times):
    """
    Print formatted results table to console.
    Format: Model | Precision | Recall | F1 | Size | Time (s)
    """
    print("\n" + "="*110)
    print("EVALUATION RESULTS SUMMARY")
    print("="*110)
    print(f"{'Model':<30} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Size':>8} {'Time (s)':>12} {'Train (s)':>12}")
    print("-"*110)

    for model_name, metrics in results.items():
        train_time = training_times.get(model_name, 0)
        print(
            f"{model_name:<30} "
            f"{metrics.precision:>10.4f} "
            f"{metrics.recall:>10.4f} "
            f"{metrics.f1_score:>10.4f} "
            f"{metrics.eval_size:>8} "
            f"{metrics.eval_time_seconds:>12.2f} "
            f"{train_time:>12.2f}"
        )

    print("="*110)

def save_results(results, training_times):
    """
    Save comprehensive results to multiple formats.
    """
    output_dir = CONFIG['output_dir']

    # 1. Save JSON results
    json_path = os.path.join(output_dir, CONFIG['results_file'])
    results_dict = {
        'experiment_info': {
            'timestamp': datetime.now().isoformat(),
            'model': CONFIG['model_name'],
            'model_alias': CONFIG['model_alias'],
            'train_samples': CONFIG['train_samples'],
            'num_training_steps': CONFIG['num_training_steps'],
        },
        'results': {}
    }

    for model_name, metrics in results.items():
        metrics.training_time_seconds = training_times.get(model_name, 0)
        results_dict['results'][model_name] = metrics.to_dict()

    with open(json_path, 'w') as f:
        json.dump(results_dict, f, indent=2)
    print(f"JSON results saved to {json_path}")

    # 2. Save summary CSV
    csv_path = os.path.join(output_dir, CONFIG['summary_csv'])
    rows = []
    for model_name, metrics in results.items():
        rows.append({
            'Model': model_name,
            'Precision': round(metrics.precision, 4),
            'Recall': round(metrics.recall, 4),
            'F1': round(metrics.f1_score, 4),
            'Size': metrics.eval_size,
            'Time (s)': round(metrics.eval_time_seconds, 2),
            'Train Time (s)': round(training_times.get(model_name, 0), 2),
        })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    print(f"Summary CSV saved to {csv_path}")

def create_comparison_plot(results):
    """
    Create bar chart comparing model performances.
    """
    output_path = os.path.join(CONFIG['output_dir'], 'model_comparison.png')

    models = list(results.keys())

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = ['#ff6b6b', '#4ecdc4', '#45b7d1'][:len(models)]

    metrics_data = [
        (axes[0, 0], [results[m].accuracy for m in models], 'Accuracy'),
        (axes[0, 1], [results[m].precision for m in models], 'Precision'),
        (axes[1, 0], [results[m].recall for m in models], 'Recall'),
        (axes[1, 1], [results[m].f1_score for m in models], 'F1 Score'),
    ]

    for ax, data, title in metrics_data:
        bars = ax.bar(models, data, color=colors, alpha=0.8)
        ax.set_ylabel(title, fontsize=12)
        ax.set_title(f'{title} Comparison', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1])
        ax.grid(axis='y', alpha=0.3)

        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Comparison plot saved to {output_path}")

print("Results functions defined!")

Results functions defined!


## Model Selection and Initialization

In [ ]:
model_name = CONFIG['model_name']
bnb_config = create_bnb_config()

print(f"SELECTED MODEL: {CONFIG['model_alias']}")
print(f"Model path: {model_name}")

# Initialize results storage
results = {}
training_times = {}

SELECTED MODEL: LLaMA-3.2-1B
Model path: meta-llama/Llama-3.2-1B-Instruct


## Step 1: Evaluate Base Model

In [ ]:
if not CONFIG['skip_base_eval']:
    print("\n" + "="*70)
    print("STEP 1: Evaluating BASE MODEL")
    print("="*70)

    base_model, base_tokenizer = load_model(model_name, bnb_config)
    base_results = evaluate_model(base_model, base_tokenizer, test_df, max_samples=CONFIG['eval_samples'])

    print(f"\nBASE MODEL RESULTS:")
    print(f"   Precision: {base_results.precision:.4f}")
    print(f"   Recall: {base_results.recall:.4f}")
    print(f"   F1 Score: {base_results.f1_score:.4f}")
    print(f"   Time: {base_results.eval_time_seconds:.2f}s")

    results['Base Model'] = base_results
    training_times['Base Model'] = 0.0

    del base_model
    torch.cuda.empty_cache()
else:
    print("Skipping base model evaluation (skip_base_eval=True)")

## Step 2: Fine-tune WITHOUT Rules (Baseline)

In [ ]:
if not CONFIG['skip_baseline']:
    print("\n" + "="*70)
    print("STEP 2: FINE-TUNING WITHOUT RULES (Baseline)")
    print("="*70)

    ft_model_baseline, ft_tokenizer_baseline = load_model(model_name, bnb_config)

    train_dataset_baseline = Dataset.from_pandas(train_df_without_rules)
    train_dataset_baseline = train_dataset_baseline.map(
        lambda batch: preprocess_batch(batch, ft_tokenizer_baseline),
        batched=True
    )

    output_dir_baseline = os.path.join(CONFIG['output_dir'], "finetuned_baseline")
    ft_model_baseline, train_time_baseline = fine_tune_model(
        ft_model_baseline, ft_tokenizer_baseline,
        train_dataset_baseline, output_dir_baseline,
        num_steps=CONFIG['num_training_steps']
    )

    print("\nEVALUATING FINE-TUNED MODEL (WITHOUT RULES)")
    ft_results_baseline = evaluate_model(
        ft_model_baseline, ft_tokenizer_baseline,
        test_df, max_samples=CONFIG['eval_samples']
    )

    print(f"\nBASELINE RESULTS:")
    print(f"   Precision: {ft_results_baseline.precision:.4f}")
    print(f"   Recall: {ft_results_baseline.recall:.4f}")
    print(f"   F1 Score: {ft_results_baseline.f1_score:.4f}")
    print(f"   Eval Time: {ft_results_baseline.eval_time_seconds:.2f}s")
    print(f"   Train Time: {train_time_baseline:.2f}s")

    results['Baseline'] = ft_results_baseline
    training_times['Baseline'] = train_time_baseline

    del ft_model_baseline
    torch.cuda.empty_cache()
else:
    print("Skipping baseline training (skip_baseline=True)")

## Step 3: Fine-tune WITH Rules (MetaMine)

In [ ]:
print("\n" + "="*70)
print("STEP 3: FINE-TUNING WITH RULES (MetaMine)")
print("="*70)

ft_model_rules, ft_tokenizer_rules = load_model(model_name, bnb_config)

train_dataset_rules = Dataset.from_pandas(train_df_with_rules)
train_dataset_rules = train_dataset_rules.map(
    lambda batch: preprocess_batch(batch, ft_tokenizer_rules),
    batched=True
)

output_dir_rules = os.path.join(CONFIG['output_dir'], "finetuned_metamine")
ft_model_rules, train_time_rules = fine_tune_model(
    ft_model_rules, ft_tokenizer_rules,
    train_dataset_rules, output_dir_rules,
    num_steps=CONFIG['num_training_steps']
)

print("\nEVALUATING FINE-TUNED MODEL (WITH RULES / MetaMine)")
ft_results_rules = evaluate_model(
    ft_model_rules, ft_tokenizer_rules,
    test_df, max_samples=CONFIG['eval_samples']
)
 8
print(f"\nMETAMINE RESULTS:")
print(f"   Precision: {ft_results_rules.precision:.4f}")
print(f"   Recall: {ft_results_rules.recall:.4f}")
print(f"   F1 Score: {ft_results_rules.f1_score:.4f}")
print(f"   Eval Time: {ft_results_rules.eval_time_seconds:.2f}s")
print(f"   Train Time: {train_time_rules:.2f}s")

results['METAMINE'] = ft_results_rules
training_times['METAMINE'] = train_time_rules


STEP 3: FINE-TUNING WITH RULES (MetaMine)
Loading model meta-llama/Llama-3.2-1B-Instruct with 1 GPU(s)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

...

Step,Training Loss
50,0.561200
100,0.236800
150,0.220900
200,0.220400
250,0.219500
300,0.228800
350,0.222900
400,0.218400
450,0.216300
500,0.224100


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Training completed in 631.80 seconds
Saving model to /content/drive/MyDrive/KG-LLM/KG/LDM/output/finetuned_metamine...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



EVALUATING FINE-TUNED MODEL (WITH RULES / MetaMine)
Evaluating on 1000 samples...
Processed 50/1000 samples... (479.2s elapsed)
Processed 100/1000 samples... (958.1s elapsed)
Processed 150/1000 samples... (1431.5s elapsed)
Processed 200/1000 samples... (1885.5s elapsed)
Processed 250/1000 samples... (2349.8s elapsed)
Processed 300/1000 samples... (2829.9s elapsed)
Processed 350/1000 samples... (3298.6s elapsed)
Processed 400/1000 samples... (3780.7s elapsed)
Processed 450/1000 samples... (4247.5s elapsed)
Processed 500/1000 samples... (4717.6s elapsed)
Processed 550/1000 samples... (5179.8s elapsed)
Processed 600/1000 samples... (5677.3s elapsed)
Processed 650/1000 samples... (6168.3s elapsed)
Processed 700/1000 samples... (6661.3s elapsed)
Processed 750/1000 samples... (7162.1s elapsed)
Processed 800/1000 samples... (7655.2s elapsed)
Processed 850/1000 samples... (8156.1s elapsed)
Processed 900/1000 samples... (8642.5s elapsed)
Processed 950/1000 samples... (9141.3s elapsed)
Processe

## Final Results Summary

In [ ]:
# Print results table
print_results_table(results, training_times)

# Save all results
save_results(results, training_times)

# Generate comparison plot
if CONFIG['generate_plots']:
    create_comparison_plot(results)

## View Saved Results

In [ ]:
# Display the saved CSV results
print("\n" + "="*70)
print("SAVED RESULTS FILES")
print("="*70)

# Summary CSV
summary_path = os.path.join(CONFIG['output_dir'], CONFIG['summary_csv'])
print(f"\nResults Summary ({summary_path}):")
print("-"*70)
summary_df = pd.read_csv(summary_path)
print(summary_df.to_string(index=False))

print("\n" + "="*70)
print(f"All results saved to: {CONFIG['output_dir']}")
print("="*70)